# Explore "fintech_data"

Generated for a selected volume/folder in Catalog Explorer. This notebook shows how to do the following: - viewing file metadata - filtering files by type - filtering files by size and date - processing files with AI functions - setting up Auto Loader

In [0]:
%sql
-- 1. Create Raw Transactions Table
CREATE OR REPLACE TABLE raw_transactions AS
SELECT * FROM read_files('/Volumes/workspace/default/fintech_data/archive (1)/transactions.csv', format => 'csv', header => 'true', inferSchema => 'true');

-- 2. Create Raw Users Table
CREATE OR REPLACE TABLE raw_users AS
SELECT * FROM read_files('/Volumes/workspace/default/fintech_data/archive (1)/users.csv', format => 'csv', header => 'true', inferSchema => 'true');

-- 3. Create Raw Merchants Table
CREATE OR REPLACE TABLE raw_merchants AS
SELECT * FROM read_files('/Volumes/workspace/default/fintech_data/archive (1)/merchants.csv', format => 'csv', header => 'true', inferSchema => 'true');

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM raw_merchants LIMIT 5;

merchant_id,merchant_name,category,city,region,mcc_code,risk_score,avg_transaction_uzs,is_online,years_registered,_rescued_data
M00000,Nur Store,Fuel,Jizzakh,Jizzakh,8073,4.1,288152,1,2,null
M00001,Fayz Plus,Telecom,Guliston,Sirdaryo,4443,20.2,190779,0,8,null
M00002,Mehr Market,Utilities,Nukus,Karakalpakstan,2872,12.7,37694,1,3,null
M00003,Shams Bozor,Clothing,Andijan,Andijan,4679,2.1,439098,0,8,null
M00004,Fayz Center,Grocery,Fergana,Fergana,3205,18.5,96090,0,2,null


In [0]:
%sql
CREATE OR REPLACE TABLE enriched_transactions AS

-- Step 1: Compute statistical baselines (Mean and Standard Deviation per Customer)
WITH CustomerStats AS (
    SELECT 
        user_id, 
        AVG(amount_uzs) AS Mean_Amount,
        COALESCE(STDDEV(amount_uzs), 0) AS StdDev_Amount
    FROM raw_transactions
    WHERE amount_uzs IS NOT NULL AND user_id IS NOT NULL
    GROUP BY user_id
),

-- Step 2: Calculate Z-Scores and deviations for each transaction
TransactionZScores AS (
    SELECT 
        t.*,
        c.Mean_Amount,
        c.StdDev_Amount,
        CASE 
            WHEN c.StdDev_Amount = 0 THEN 0 
            ELSE (t.amount_uzs - c.Mean_Amount) / c.StdDev_Amount 
        END AS Z_Score
    FROM raw_transactions t
    LEFT JOIN CustomerStats c ON t.user_id = c.user_id
)

-- Step 3: Combine with User and Merchant attributes, and apply statistical Risk Tiers
SELECT 
    tz.tx_id,
    tz.timestamp,
    tz.amount_uzs,
    tz.channel,
    tz.is_fraud,
    -- User Attributes
    u.user_id,
    u.region AS User_Region,
    -- Merchant Attributes
    m.merchant_id,
    m.category AS MerchantCategory,
    -- Statistical Metrics
    ROUND(tz.Mean_Amount, 2) AS Customer_Mean_Amount,
    ROUND(tz.StdDev_Amount, 2) AS Customer_StdDev_Amount,
    ROUND(tz.Z_Score, 2) AS Amount_Z_Score,
    ROUND(tz.amount_uzs - tz.Mean_Amount, 2) AS Amount_Deviation,
    -- Statistical Risk Tiering based on Z-Scores
    CASE 
        WHEN tz.Z_Score > 3.0 THEN 'High Risk (Statistical Outlier)'
        WHEN tz.Z_Score > 2.0 THEN 'Medium Risk (Moderate Deviation)'
        ELSE 'Low Risk (Normal Behavior)'
    END AS Risk_Tier
FROM TransactionZScores tz
LEFT JOIN raw_users u ON tz.user_id = u.user_id
LEFT JOIN raw_merchants m ON tz.merchant_id = m.merchant_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT risk_tier, COUNT(*) AS transaction_count, ROUND(AVG(amount_uzs), 2) AS avg_amount
FROM enriched_transactions
GROUP BY risk_tier;

risk_tier,transaction_count,avg_amount
Low Risk (Normal Behavior),140394,205037.84
High Risk (Statistical Outlier),2065,1992908.64
Medium Risk (Moderate Deviation),7541,1076892.96
